<img src="https://fedesarrollo.org.co/logo-Fede.png" alt="Fedesarrollo" style="float: right; width: 300px; height: auto;">
<img src="https://upload.wikimedia.org/wikipedia/commons/f/f9/Logo_BID_Espa%C3%B1ol.png" alt="Banco Interamericano de Desarrollo" style="float: right; width: 220px; height: auto;">

# Deflactor del producto interno bruto de Estados Unidos

Editor: Juan Diego Heredia Niño - jheredia@fedesarrollo.org.co

Septiembre 2026

### Importar librerías

La siguiente celda marca el inicio de un nuevo bloque de trabajo independiente:

**Librerías utilizadas:**
- `pandas`: para manipulación y análisis de datos estructurados.

In [1]:
import pandas as pd

### Configuración de rutas

Carga las rutas del proyecto y las opciones globales desde `config.py`, instalado como
paquete editable en la raíz del proyecto.

In [2]:
from config import raw, temp

### Parámetros

El deflactor se obtiene de la serie `GDPDEF` de la Reserva Federal de San Luis, que es una
fuente viva: el proveedor revisa el histórico hacia atrás con cada publicación. La descarga
original es del 25 de agosto de 2025 y su salida está archivada en `data/raw/descargas/`.

La bandera `reejecutar_descarga` queda en falso para que la cadena reproduzca esa misma serie.
Al ponerla en verdadero se vuelve a consultar la API y el resultado puede diferir del original.

In [3]:
reejecutar_descarga = False

fecha_inicio = '1990-01-01'
fecha_fin = '2024-12-31'
anio_base = 2019

### Construcción de la serie anual

Promedia la serie trimestral a frecuencia anual y la expresa como factor relativo al año base,
de modo que multiplicar por el factor lleva los valores corrientes a precios del año base.

In [4]:
if reejecutar_descarga:
    import pandas_datareader.data as web

    gdp_def_q = web.DataReader('GDPDEF', 'fred', fecha_inicio, fecha_fin)

    gdp_def_a = gdp_def_q.resample('YE').mean()
    gdp_def_a.index = gdp_def_a.index.year

    gdp_def_base = gdp_def_a.loc[anio_base, 'GDPDEF']
    gdp_def_a['deflactor_2019'] = gdp_def_base / gdp_def_a['GDPDEF']
    gdp_def_a.drop(columns=['GDPDEF'], inplace=True)

    gdp_def_a.reset_index(inplace=True)
    gdp_def_a.rename(columns={'DATE': 'anio'}, inplace=True)
else:
    gdp_def_a = pd.read_parquet(raw / 'descargas' / 'deflactor.parquet')

(temp / 'exportaciones').mkdir(parents=True, exist_ok=True)
gdp_def_a.to_parquet(temp / 'exportaciones' / 'deflactor.parquet', index=True)
print(f"años en la serie: {len(gdp_def_a)}")

años en la serie: 35
